# EN↔FR Phrase Alignment — Colab (GPU)

Run the cells in order. For the **highest accuracy** use a GPU runtime:
**Runtime → Change runtime type → Hardware accelerator: T4 GPU**.

CPU-only runtimes also work: the framework falls back to its pure rule-based
engine automatically (`--method auto`).

## 0) Install dependencies
Downloads torch + transformers + awesome-align. On the first run the mBERT
model (~680 MB) is downloaded once and then cached.

In [ ]:
%pip install -q torch transformers numpy awesome-align==2.0.1

## 1) Get your CSV
Choose one:
- **Upload from your computer** (next cell), or
- **Mount Google Drive** and set `SRC` to the file path (commented-out
  alternative in the next cell).

In [ ]:
from google.colab import files
print('Select your CSV (columns: English, French):')
uploaded = files.upload()
SRC = list(uploaded.keys())[0]
print('SRC =', SRC)

# Option B — Google Drive instead of uploading:
# from google.colab import drive
# drive.mount('/content/drive')
# SRC = '/content/drive/MyDrive/path/to/your_file.csv'

## 2) Get the framework
Either clone your GitHub repo, or upload the `en_fr_align` folder as a zip
(commented-out alternative below).

In [ ]:
!git clone https://github.com/YOUR_USERNAME/en_fr_align.git
%cd en_fr_align

# Option B — upload the framework zip instead:
# up = files.upload()
# !unzip -q {list(up.keys())[0]}
# %cd en_fr_align

## 3) Run the alignment (GPU)
Adds the `Alignment` column with one bullet per phrase/clause/punctuation
unit. Progress prints every 2,048 rows. For a quick test first, append
`--limit 200` to the command.

In [ ]:
import os
OUT = os.path.splitext(os.path.basename(SRC))[0] + '_aligned.csv'
!python run_align.py --src '{SRC}' --out '{OUT}' --method neural --batch-size 32
print('Done:', OUT)

## 4) Quality check + preview
Runs the 14 automated QC checks (row count, unchanged EN/FR columns,
valid multiline CSV, no column spill, etc.).

In [ ]:
!python run_align.py --src '{SRC}' --out '{OUT}' --check
print()
print('Preview (first 6 rows):')
import csv
with open(OUT, encoding='utf-8') as f:
    for i, row in enumerate(csv.reader(f)):
        if i >= 6:
            break
        print('--- row', i, '---')
        print(row[0][:90])
        print(row[1][:90])
        print(row[2].split('\n')[0][:130])

## 5) Download the result
Downloads `<name>_aligned.csv` to your computer. To keep a copy on Drive
instead, uncomment the last line.

In [ ]:
from google.colab import files
files.download(OUT)
# !cp '{OUT}' '/content/drive/MyDrive/'